# PowerNZ Kaggle Simple

Este notebook tiene una sola celda grande para evitar errores de orden.

## Antes De Ejecutar

1. En Kaggle, subo el video como `Input` del notebook.
2. En `Settings`, activo `Internet`.
3. Si aparece, pongo `Accelerator` en GPU.
4. Cambio `EXERCISE` en la celda de abajo.
5. Ejecuto la celda.

El resultado queda en `/kaggle/working/powernz_analizado.mp4`.

In [ ]:
# CAMBIO SOLO ESTO
EXERCISE = "deadlift"  # deadlift, squat o bench
PLATE_DIAMETER_PX = 120

# NO TOCAR DESDE AQUI
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/juanrdzmb/PowerNZ.git"
BRANCH = "codex/model-download-and-launcher"
OUTPUT_NAME = "powernz_analizado.mp4"

def run(command):
    print("\n$", " ".join(str(part) for part in command))
    subprocess.run([str(part) for part in command], check=True)

print("PowerNZ Kaggle Simple")
print("Ejercicio:", EXERCISE)
print("Python:", sys.version)

workdir = Path("/kaggle/working")
repo_dir = workdir / "PowerNZ"

if repo_dir.exists():
    shutil.rmtree(repo_dir)

run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, repo_dir])
os.chdir(repo_dir)

# Kaggle ya trae muchas librerias; aun asi instalo las que falten.
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

# Descarga powerai_bar_detector.pt y powerai_athlete_seg.pt desde Hugging Face.
run([sys.executable, "model_downloader.py"])

videos = []
for pattern in ("*.mp4", "*.mov", "*.avi", "*.mkv"):
    videos.extend(Path("/kaggle/input").rglob(pattern))
videos = sorted(videos)

if not videos:
    raise FileNotFoundError("No encontre videos en /kaggle/input. Asegurate de subir el video como Input del notebook.")

print("\nVideos encontrados:")
for index, path in enumerate(videos):
    print(index, path)

video_path = videos[0]
output_path = workdir / OUTPUT_NAME
print("\nAnalizando:", video_path)
print("Salida:", output_path)

run([
    sys.executable, "main.py",
    "--input", video_path,
    "--output", output_path,
    "--exercise", EXERCISE,
    "--pose-backend", "yolo",
    "--plate-diameter-px", PLATE_DIAMETER_PX,
    "--output-format", "portrait-720",
    "--no-mobile-conversion",
])

print("\nLISTO. Descarga este archivo desde Output:")
print(output_path)